# **LABORATORIO DIRIGIDO N° 03**
## **Fundamentos de Gestión de Datos**
### Semana 3: SQL Avanzado - Joins y Agrupaciones

---

**Docente:** Pilar Rocío Sayán Mejía  
**Curso:** Fundamentos de Gestión de Datos  
**Semestre:** 2026-I  

---

**Tema:** Análisis de Ventas con JOINs en SQLite  

**Objetivo:** Aplicar consultas SQL avanzadas utilizando JOINs (INNER, LEFT, RIGHT, FULL OUTER), agrupaciones con GROUP BY y HAVING, subconsultas y vistas para analizar relaciones entre múltiples tablas de una base de datos de ventas.

## **Actividad 1: Revisión de Conceptos**

Complete las siguientes definiciones con sus propias palabras:

**1. ¿Qué es un JOIN y para qué se utiliza en SQL?**

**Respuesta:**

_Escribe tu respuesta aquí..._

**2. Diferencia entre INNER JOIN y LEFT JOIN**

**Respuesta:**

_Escribe tu respuesta aquí..._

**3. ¿Qué es RIGHT JOIN y FULL OUTER JOIN?**

**Respuesta:**

_Escribe tu respuesta aquí..._

**4. ¿Qué hace la cláusula GROUP BY?**

**Respuesta:**

_Escribe tu respuesta aquí..._

**5. Diferencia entre WHERE y HAVING**

**Respuesta:**

_Escribe tu respuesta aquí..._

**6. ¿Qué es una subconsulta y cuándo se usa?**

**Respuesta:**

_Escribe tu respuesta aquí..._

**7. ¿Qué es una vista (VIEW) y sus ventajas?**

**Respuesta:**

_Escribe tu respuesta aquí..._

**8. Funciones de agregación: COUNT, SUM, AVG, MAX, MIN**

**Respuesta:**

_Escribe tu respuesta aquí..._

**9. ¿Qué es un alias en SQL y para qué sirve?**

**Respuesta:**

_Escribe tu respuesta aquí..._

**10. Buenas prácticas de legibilidad en consultas SQL**

**Respuesta:**

_Escribe tu respuesta aquí..._

---
## **Actividad 2: Desarrollo Práctico - Análisis de Ventas con JOINs**

### **Paso 0: Importar librerías y crear la base de datos**

In [ ]:
import sqlite3
import pandas as pd

DB_FILE = "ventas_lab.db"

# ─── Configuración de la base de datos ──────────────────────────────────────

def setup_database():
    """Crea las tablas e inserta los datos de ejemplo.

    Es IDEMPOTENTE: ejecutarla varias veces no duplica ni sobreescribe datos.
    Se llama automáticamente desde sql() — no es necesario ejecutarla a mano.
    """
    with sqlite3.connect(DB_FILE) as conn:
        # CREATE TABLE IF NOT EXISTS — no falla si las tablas ya existen
        conn.executescript("""
            CREATE TABLE IF NOT EXISTS clientes (
                id_cliente INTEGER PRIMARY KEY,
                nombre     TEXT NOT NULL,
                ciudad     TEXT,
                email      TEXT
            );
            CREATE TABLE IF NOT EXISTS productos (
                id_producto     INTEGER PRIMARY KEY,
                nombre_producto TEXT NOT NULL,
                categoria       TEXT,
                precio          REAL
            );
            CREATE TABLE IF NOT EXISTS pedidos (
                id_pedido  INTEGER PRIMARY KEY,
                id_cliente INTEGER,
                fecha      TEXT,
                total      REAL,
                FOREIGN KEY (id_cliente) REFERENCES clientes(id_cliente)
            );
            CREATE TABLE IF NOT EXISTS detalles_pedido (
                id_detalle      INTEGER PRIMARY KEY,
                id_pedido       INTEGER,
                id_producto     INTEGER,
                cantidad        INTEGER,
                precio_unitario REAL,
                FOREIGN KEY (id_pedido)   REFERENCES pedidos(id_pedido),
                FOREIGN KEY (id_producto) REFERENCES productos(id_producto)
            );
        """)

        # Insertar datos solo si las tablas están vacías
        if conn.execute("SELECT COUNT(*) FROM clientes").fetchone()[0] == 0:
            clientes = [
                (1,  'María García',     'Lima',     'maria@email.com'),
                (2,  'Juan Pérez',       'Arequipa', 'juan@email.com'),
                (3,  'Ana López',        'Cusco',    'ana@email.com'),
                (4,  'Carlos Rodríguez', 'Lima',     'carlos@email.com'),
                (5,  'Laura Martínez',   'Trujillo', 'laura@email.com'),
                (6,  'Pedro Sánchez',    'Lima',     'pedro@email.com'),
                (7,  'Rosa Torres',      'Arequipa', 'rosa@email.com'),
                (8,  'Miguel Flores',    'Cusco',    'miguel@email.com'),
                (9,  'Carmen Ruiz',      'Lima',     'carmen@email.com'),
                (10, 'José Díaz',        'Trujillo', 'jose@email.com'),
            ]
            conn.executemany('INSERT INTO clientes VALUES (?, ?, ?, ?)', clientes)

            productos = [
                (1,  'Laptop HP',        'Electrónica', 2500.00),
                (2,  'Mouse Logitech',   'Electrónica',   80.00),
                (3,  'Teclado Mecánico', 'Electrónica',  150.00),
                (4,  'Monitor Samsung',  'Electrónica',  800.00),
                (5,  'Silla Ergonómica', 'Oficina',      450.00),
                (6,  'Escritorio',       'Oficina',      600.00),
                (7,  'Impresora Epson',  'Electrónica',  350.00),
                (8,  'Auriculares Sony', 'Electrónica',  200.00),
                (9,  'Webcam Logitech',  'Electrónica',  120.00),
                (10, 'Parlantes JBL',    'Electrónica',  180.00),
            ]
            conn.executemany('INSERT INTO productos VALUES (?, ?, ?, ?)', productos)

            pedidos = [
                ( 1, 1, '2026-01-15', 2650.00), ( 2, 2, '2026-01-16',  950.00),
                ( 3, 1, '2026-01-20',  450.00), ( 4, 3, '2026-01-22', 3300.00),
                ( 5, 4, '2026-01-25',  150.00), ( 6, 1, '2026-02-01',  800.00),
                ( 7, 5, '2026-02-03', 1050.00), ( 8, 6, '2026-02-05',  450.00),
                ( 9, 2, '2026-02-08',  200.00), (10, 7, '2026-02-10', 1050.00),
                (11, 4, '2026-02-12',  430.00), (12, 9, '2026-02-15', 2700.00),
                (13, 1, '2026-02-18',  380.00), (14, 3, '2026-02-20',  150.00),
            ]
            conn.executemany('INSERT INTO pedidos VALUES (?, ?, ?, ?)', pedidos)

            detalles = [
                ( 1, 1, 1, 1, 2500.00), ( 2, 1, 2, 2,   80.00),
                ( 3, 2, 4, 1,  800.00), ( 4, 2, 5, 1,  450.00),
                ( 5, 3, 5, 1,  450.00),
                ( 6, 4, 1, 1, 2500.00), ( 7, 4, 4, 1,  800.00),
                ( 8, 5, 3, 1,  150.00),
                ( 9, 6, 4, 1,  800.00),
                (10, 7, 3, 3,  150.00), (11, 7, 8, 3,  200.00),
                (12, 8, 5, 1,  450.00),
                (13, 9, 8, 1,  200.00),
                (14,10, 3, 3,  150.00), (15,10,10, 3,  180.00),
                (16,11, 2, 2,   80.00), (17,11, 9, 2,  120.00),
                (18,12, 1, 1, 2500.00), (19,12, 8, 1,  200.00),
                (20,13, 9, 2,  120.00), (21,13,10, 1,  180.00),
                (22,14, 3, 1,  150.00),
            ]
            conn.executemany('INSERT INTO detalles_pedido VALUES (?, ?, ?, ?, ?)', detalles)
            print(f"📥 Datos cargados en '{DB_FILE}'")
        else:
            print(f"ℹ️  Base de datos '{DB_FILE}' ya inicializada — no se reinsertan datos.")


def sql(query, params=None):
    """Ejecuta una consulta SQL y devuelve un DataFrame de pandas.

    Garantiza automáticamente que la BD esté lista — funciona en cualquier celda,
    incluso después de reiniciar el kernel.

    Uso básico:   df = sql("SELECT * FROM clientes")
    Con params:   df = sql("SELECT * FROM clientes WHERE ciudad = ?", params=["Lima"])
    """
    setup_database()  # idempotente: no hace nada si la BD ya existe
    with sqlite3.connect(DB_FILE) as conn:
        return pd.read_sql_query(query, conn, params=params)


# Inicializar al cargar el notebook
setup_database()
print("\n💡 Helper listo → sql('SELECT ...') funciona en cualquier celda del notebook.")

### **Base de datos persistente: `ventas_lab.db`**

La función `setup_database()` (ejecutada automáticamente arriba) creó las 4 tablas e insertó los datos.

- **Archivo en disco** → persiste aunque reinicies el kernel de Jupyter.
- **Idempotente** → volver a ejecutar la celda anterior no duplica datos.
- **Independencia de celdas** → `sql('SELECT ...')` funciona en cualquier celda, incluso si saltas el orden.

> **Si ves errores inesperados:** ejecuta la celda del *Paso 0* y continúa desde donde estabas.

**Tablas disponibles:** `clientes` · `productos` · `pedidos` · `detalles_pedido`

In [3]:
# Insertar datos de clientes
clientes = [
    (1, 'María García', 'Lima', 'maria@email.com'),
    (2, 'Juan Pérez', 'Arequipa', 'juan@email.com'),
    (3, 'Ana López', 'Cusco', 'ana@email.com'),
    (4, 'Carlos Rodríguez', 'Lima', 'carlos@email.com'),
    (5, 'Laura Martínez', 'Trujillo', 'laura@email.com'),
    (6, 'Pedro Sánchez', 'Lima', 'pedro@email.com'),
    (7, 'Rosa Torres', 'Arequipa', 'rosa@email.com'),
    (8, 'Miguel Flores', 'Cusco', 'miguel@email.com'),
    (9, 'Carmen Ruiz', 'Lima', 'carmen@email.com'),
    (10, 'José Díaz', 'Trujillo', 'jose@email.com')
]
cursor.executemany('INSERT INTO clientes VALUES (?, ?, ?, ?)', clientes)

# Insertar datos de productos
productos = [
    (1, 'Laptop HP', 'Electrónica', 2500.00),
    (2, 'Mouse Logitech', 'Electrónica', 80.00),
    (3, 'Teclado Mecánico', 'Electrónica', 150.00),
    (4, 'Monitor Samsung', 'Electrónica', 800.00),
    (5, 'Silla Ergonómica', 'Oficina', 450.00),
    (6, 'Escritorio', 'Oficina', 600.00),
    (7, 'Impresora Epson', 'Electrónica', 350.00),
    (8, 'Auriculares Sony', 'Electrónica', 200.00),
    (9, 'Webcam Logitech', 'Electrónica', 120.00),
    (10, 'Parlantes JBL', 'Electrónica', 180.00)
]
cursor.executemany('INSERT INTO productos VALUES (?, ?, ?, ?)', productos)

# Insertar datos de pedidos
pedidos = [
    (1, 1, '2026-01-15', 2650.00),
    (2, 2, '2026-01-16', 950.00),
    (3, 1, '2026-01-20', 450.00),
    (4, 3, '2026-01-22', 3300.00),
    (5, 4, '2026-01-25', 150.00),
    (6, 1, '2026-02-01', 800.00),
    (7, 5, '2026-02-03', 1050.00),
    (8, 6, '2026-02-05', 450.00),
    (9, 2, '2026-02-08', 200.00),
    (10, 7, '2026-02-10', 1050.00),
    (11, 4, '2026-02-12', 430.00),
    (12, 9, '2026-02-15', 2700.00),
    (13, 1, '2026-02-18', 380.00),
    (14, 3, '2026-02-20', 150.00)
]
cursor.executemany('INSERT INTO pedidos VALUES (?, ?, ?, ?)', pedidos)

# Insertar datos de detalles_pedido
detalles = [
    (1, 1, 1, 1, 2500.00), (2, 1, 2, 2, 80.00),
    (3, 2, 4, 1, 800.00), (4, 2, 5, 1, 450.00),
    (5, 3, 5, 1, 450.00),
    (6, 4, 1, 1, 2500.00), (7, 4, 4, 1, 800.00),
    (8, 5, 3, 1, 150.00),
    (9, 6, 4, 1, 800.00),
    (10, 7, 3, 3, 150.00), (11, 7, 8, 3, 200.00),
    (12, 8, 5, 1, 450.00),
    (13, 9, 8, 1, 200.00),
    (14, 10, 3, 3, 150.00), (15, 10, 10, 3, 180.00),
    (16, 11, 2, 2, 80.00), (17, 11, 9, 2, 120.00),
    (18, 12, 1, 1, 2500.00), (19, 12, 8, 1, 200.00),
    (20, 13, 9, 2, 120.00), (21, 13, 10, 1, 180.00),
    (22, 14, 3, 1, 150.00)
]
cursor.executemany('INSERT INTO detalles_pedido VALUES (?, ?, ?, ?, ?)', detalles)

conn.commit()
print("✅ Datos insertados exitosamente")

✅ Datos insertados exitosamente


In [4]:
# Ver todas las tablas
query = "SELECT name FROM sqlite_master WHERE type='table';"
tablas = pd.read_sql_query(query, conn)
print("📋 TABLAS EN LA BASE DE DATOS:")
print(tablas)

📋 TABLAS EN LA BASE DE DATOS:
              name
0         clientes
1        productos
2          pedidos
3  detalles_pedido


In [5]:
# Explorar estructura de cada tabla
for tabla in ['clientes', 'productos', 'pedidos', 'detalles_pedido']:
    print(f"\n📋 TABLA: {tabla.upper()}")
    estructura = pd.read_sql_query(f"PRAGMA table_info({tabla});", conn)
    print(estructura)


📋 TABLA: CLIENTES
   cid        name     type  notnull dflt_value  pk
0    0  id_cliente  INTEGER        0       None   1
1    1      nombre     TEXT        1       None   0
2    2      ciudad     TEXT        0       None   0
3    3       email     TEXT        0       None   0

📋 TABLA: PRODUCTOS
   cid             name     type  notnull dflt_value  pk
0    0      id_producto  INTEGER        0       None   1
1    1  nombre_producto     TEXT        1       None   0
2    2        categoria     TEXT        0       None   0
3    3           precio     REAL        0       None   0

📋 TABLA: PEDIDOS
   cid        name     type  notnull dflt_value  pk
0    0   id_pedido  INTEGER        0       None   1
1    1  id_cliente  INTEGER        0       None   0
2    2       fecha     TEXT        0       None   0
3    3       total     REAL        0       None   0

📋 TABLA: DETALLES_PEDIDO
   cid             name     type  notnull dflt_value  pk
0    0       id_detalle  INTEGER        0       None   

In [ ]:
# Ver todas las tablas
df = sql("SELECT name FROM sqlite_master WHERE type='table';")
print("📋 TABLAS EN LA BASE DE DATOS:")
print(df)

# Explorar estructura de cada tabla
for tabla in ['clientes', 'productos', 'pedidos', 'detalles_pedido']:
    print(f"\n📋 TABLA: {tabla.upper()}")
    print(sql(f"PRAGMA table_info({tabla});"))

# Ver primeros registros de cada tabla
print("📋 CLIENTES:")
print(sql("SELECT * FROM clientes;"))

print("\n📋 PRODUCTOS:")
print(sql("SELECT * FROM productos;"))

print("\n📋 PEDIDOS:")
print(sql("SELECT * FROM pedidos;"))

print("\n📋 DETALLES_PEDIDO:")
print(sql("SELECT * FROM detalles_pedido;"))

In [7]:
# INNER JOIN: Solo clientes que tienen pedidos
query = '''
SELECT
    c.nombre,
    c.ciudad,
    p.id_pedido,
    p.fecha,
    p.total
FROM clientes c
INNER JOIN pedidos p ON c.id_cliente = p.id_cliente
ORDER BY p.fecha DESC
LIMIT 10;
'''
df_inner = pd.read_sql_query(query, conn)
print("📋 INNER JOIN - Clientes con pedidos:")
print(df_inner)

📋 INNER JOIN - Clientes con pedidos:
             nombre    ciudad  id_pedido       fecha   total
0         Ana López     Cusco         14  2026-02-20   150.0
1      María García      Lima         13  2026-02-18   380.0
2       Carmen Ruiz      Lima         12  2026-02-15  2700.0
3  Carlos Rodríguez      Lima         11  2026-02-12   430.0
4       Rosa Torres  Arequipa         10  2026-02-10  1050.0
5        Juan Pérez  Arequipa          9  2026-02-08   200.0
6     Pedro Sánchez      Lima          8  2026-02-05   450.0
7    Laura Martínez  Trujillo          7  2026-02-03  1050.0
8      María García      Lima          6  2026-02-01   800.0
9  Carlos Rodríguez      Lima          5  2026-01-25   150.0


In [8]:
# Contar cuántos clientes tienen pedidos
query = '''
SELECT COUNT(DISTINCT c.id_cliente) as clientes_con_pedidos
FROM clientes c
INNER JOIN pedidos p ON c.id_cliente = p.id_cliente;
'''
print(pd.read_sql_query(query, conn))

   clientes_con_pedidos
0                     8


# INNER JOIN: Solo clientes que tienen pedidos
query = '''
SELECT
    c.nombre,
    c.ciudad,
    p.id_pedido,
    p.fecha,
    p.total
FROM clientes c
INNER JOIN pedidos p ON c.id_cliente = p.id_cliente
ORDER BY p.fecha DESC
LIMIT 10;
'''
df_inner = sql(query)
print("📋 INNER JOIN - Clientes con pedidos:")
print(df_inner)

# Contar cuántos clientes tienen pedidos
query = '''
SELECT COUNT(DISTINCT c.id_cliente) as clientes_con_pedidos
FROM clientes c
INNER JOIN pedidos p ON c.id_cliente = p.id_cliente;
'''
print(sql(query))

In [9]:
# LEFT JOIN: Todos los clientes, incluyendo los que no tienen pedidos
query = '''
SELECT
    c.nombre,
    c.ciudad,
    p.id_pedido,
    p.total
FROM clientes c
LEFT JOIN pedidos p ON c.id_cliente = p.id_cliente
ORDER BY c.nombre;
'''
df_left = pd.read_sql_query(query, conn)
print("📋 LEFT JOIN - Todos los clientes:")
print(df_left)

📋 LEFT JOIN - Todos los clientes:
              nombre    ciudad  id_pedido   total
0          Ana López     Cusco       14.0   150.0
1          Ana López     Cusco        4.0  3300.0
2   Carlos Rodríguez      Lima        5.0   150.0
3   Carlos Rodríguez      Lima       11.0   430.0
4        Carmen Ruiz      Lima       12.0  2700.0
5          José Díaz  Trujillo        NaN     NaN
6         Juan Pérez  Arequipa        9.0   200.0
7         Juan Pérez  Arequipa        2.0   950.0
8     Laura Martínez  Trujillo        7.0  1050.0
9       María García      Lima       13.0   380.0
10      María García      Lima        3.0   450.0
11      María García      Lima        6.0   800.0
12      María García      Lima        1.0  2650.0
13     Miguel Flores     Cusco        NaN     NaN
14     Pedro Sánchez      Lima        8.0   450.0
15       Rosa Torres  Arequipa       10.0  1050.0


In [10]:
# Identificar clientes sin pedidos
query = '''
SELECT c.nombre, c.ciudad
FROM clientes c
LEFT JOIN pedidos p ON c.id_cliente = p.id_cliente
WHERE p.id_pedido IS NULL;
'''
clientes_sin_pedidos = pd.read_sql_query(query, conn)
print("📋 Clientes SIN pedidos:")
print(clientes_sin_pedidos)

📋 Clientes SIN pedidos:
          nombre    ciudad
0  Miguel Flores     Cusco
1      José Díaz  Trujillo


# LEFT JOIN: Todos los clientes, incluyendo los que no tienen pedidos
query = '''
SELECT
    c.nombre,
    c.ciudad,
    p.id_pedido,
    p.total
FROM clientes c
LEFT JOIN pedidos p ON c.id_cliente = p.id_cliente
ORDER BY c.nombre;
'''
df_left = sql(query)
print("📋 LEFT JOIN - Todos los clientes:")
print(df_left)

# Identificar clientes sin pedidos
query = '''
SELECT c.nombre, c.ciudad
FROM clientes c
LEFT JOIN pedidos p ON c.id_cliente = p.id_cliente
WHERE p.id_pedido IS NULL;
'''
clientes_sin_pedidos = sql(query)
print("📋 Clientes SIN pedidos:")
print(clientes_sin_pedidos)

In [11]:
# GROUP BY: Total de ventas por cliente
query = '''
SELECT
    c.nombre,
    COUNT(p.id_pedido) as num_pedidos,
    SUM(p.total) as total_comprado,
    ROUND(AVG(p.total), 2) as ticket_promedio
FROM clientes c
INNER JOIN pedidos p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente
ORDER BY total_comprado DESC;
'''
df_ventas = pd.read_sql_query(query, conn)
print("📋 VENTAS POR CLIENTE:")
print(df_ventas)

📋 VENTAS POR CLIENTE:
             nombre  num_pedidos  total_comprado  ticket_promedio
0      María García            4          4280.0           1070.0
1         Ana López            2          3450.0           1725.0
2       Carmen Ruiz            1          2700.0           2700.0
3        Juan Pérez            2          1150.0            575.0
4       Rosa Torres            1          1050.0           1050.0
5    Laura Martínez            1          1050.0           1050.0
6  Carlos Rodríguez            2           580.0            290.0
7     Pedro Sánchez            1           450.0            450.0


**Pregunta 4: ¿Cuáles son los 5 clientes con mayor monto total de compras? ¿Cuál es el promedio de pedidos por cliente?**

**Respuesta:**

_Escribe tu respuesta aquí..._

# GROUP BY: Total de ventas por cliente
query = '''
SELECT
    c.nombre,
    COUNT(p.id_pedido)      as num_pedidos,
    SUM(p.total)            as total_comprado,
    ROUND(AVG(p.total), 2)  as ticket_promedio
FROM clientes c
INNER JOIN pedidos p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente
ORDER BY total_comprado DESC;
'''
df_ventas = sql(query)
print("📋 VENTAS POR CLIENTE:")
print(df_ventas)

In [12]:
# HAVING: Clientes con más de 2 pedidos
query = '''
SELECT
    c.nombre,
    COUNT(p.id_pedido) as num_pedidos,
    SUM(p.total) as total_comprado
FROM clientes c
INNER JOIN pedidos p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente
HAVING COUNT(p.id_pedido) > 2
ORDER BY total_comprado DESC;
'''
df_having = pd.read_sql_query(query, conn)
print("📋 Clientes con más de 2 pedidos:")
print(df_having)

📋 Clientes con más de 2 pedidos:
         nombre  num_pedidos  total_comprado
0  María García            4          4280.0


In [13]:
# HAVING: Clientes con total comprado mayor a 1000
query = '''
SELECT
    c.nombre,
    COUNT(p.id_pedido) as num_pedidos,
    SUM(p.total) as total_comprado
FROM clientes c
INNER JOIN pedidos p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente
HAVING SUM(p.total) > 1000
ORDER BY total_comprado DESC;
'''
print(pd.read_sql_query(query, conn))

           nombre  num_pedidos  total_comprado
0    María García            4          4280.0
1       Ana López            2          3450.0
2     Carmen Ruiz            1          2700.0
3      Juan Pérez            2          1150.0
4     Rosa Torres            1          1050.0
5  Laura Martínez            1          1050.0


# HAVING: Clientes con más de 2 pedidos
query = '''
SELECT
    c.nombre,
    COUNT(p.id_pedido) as num_pedidos,
    SUM(p.total)       as total_comprado
FROM clientes c
INNER JOIN pedidos p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente
HAVING COUNT(p.id_pedido) > 2
ORDER BY total_comprado DESC;
'''
df_having = sql(query)
print("📋 Clientes con más de 2 pedidos:")
print(df_having)

# HAVING: Clientes con total comprado mayor a 1000
query = '''
SELECT
    c.nombre,
    COUNT(p.id_pedido) as num_pedidos,
    SUM(p.total)       as total_comprado
FROM clientes c
INNER JOIN pedidos p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente
HAVING SUM(p.total) > 1000
ORDER BY total_comprado DESC;
'''
print(sql(query))

In [14]:
# JOIN de 3 tablas: Productos más vendidos
query = '''
SELECT
    pr.nombre_producto,
    pr.categoria,
    SUM(d.cantidad) as total_vendido,
    ROUND(SUM(d.cantidad * d.precio_unitario), 2) as ingresos
FROM productos pr
INNER JOIN detalles_pedido d ON pr.id_producto = d.id_producto
INNER JOIN pedidos p ON d.id_pedido = p.id_pedido
GROUP BY pr.id_producto
ORDER BY ingresos DESC
LIMIT 10;
'''
df_productos = pd.read_sql_query(query, conn)
print("📋 PRODUCTOS MÁS VENDIDOS:")
print(df_productos)

📋 PRODUCTOS MÁS VENDIDOS:
    nombre_producto    categoria  total_vendido  ingresos
0         Laptop HP  Electrónica              3    7500.0
1   Monitor Samsung  Electrónica              3    2400.0
2  Silla Ergonómica      Oficina              3    1350.0
3  Teclado Mecánico  Electrónica              8    1200.0
4  Auriculares Sony  Electrónica              5    1000.0
5     Parlantes JBL  Electrónica              4     720.0
6   Webcam Logitech  Electrónica              4     480.0
7    Mouse Logitech  Electrónica              4     320.0


**Pregunta 6: ¿Cuáles son los 3 productos que generan más ingresos? ¿A qué categoría pertenecen?**

**Respuesta:**

_Escribe tu respuesta aquí..._

# JOIN de 3 tablas: Productos más vendidos
query = '''
SELECT
    pr.nombre_producto,
    pr.categoria,
    SUM(d.cantidad)                          as total_vendido,
    ROUND(SUM(d.cantidad * d.precio_unitario), 2) as ingresos
FROM productos pr
INNER JOIN detalles_pedido d ON pr.id_producto = d.id_producto
INNER JOIN pedidos p         ON d.id_pedido    = p.id_pedido
GROUP BY pr.id_producto
ORDER BY ingresos DESC
LIMIT 10;
'''
df_productos = sql(query)
print("📋 PRODUCTOS MÁS VENDIDOS:")
print(df_productos)

In [15]:
# Primero ver el promedio de ventas por pedido
query = "SELECT ROUND(AVG(total), 2) as promedio FROM pedidos;"
promedio = pd.read_sql_query(query, conn)
print("📊 Promedio de ventas por pedido:")
print(promedio)

📊 Promedio de ventas por pedido:
   promedio
0   1050.71


In [16]:
# Subconsulta: Clientes con compras above del promedio
query = '''
SELECT
    c.nombre,
    COUNT(p.id_pedido) as num_pedidos,
    ROUND(SUM(p.total), 2) as total_comprado
FROM clientes c
INNER JOIN pedidos p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente
HAVING SUM(p.total) > (SELECT AVG(total) FROM pedidos)
ORDER BY total_comprado DESC;
'''
df_subconsulta = pd.read_sql_query(query, conn)
print("📋 Clientes con compras above del promedio:")
print(df_subconsulta)

📋 Clientes con compras above del promedio:
         nombre  num_pedidos  total_comprado
0  María García            4          4280.0
1     Ana López            2          3450.0
2   Carmen Ruiz            1          2700.0
3    Juan Pérez            2          1150.0


# Primero ver el promedio de ventas por pedido
query = "SELECT ROUND(AVG(total), 2) as promedio FROM pedidos;"
promedio = sql(query)
print("📊 Promedio de ventas por pedido:")
print(promedio)

# Subconsulta: Clientes con compras above del promedio
query = '''
SELECT
    c.nombre,
    COUNT(p.id_pedido)       as num_pedidos,
    ROUND(SUM(p.total), 2)   as total_comprado
FROM clientes c
INNER JOIN pedidos p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente
HAVING SUM(p.total) > (SELECT AVG(total) FROM pedidos)
ORDER BY total_comprado DESC;
'''
df_subconsulta = sql(query)
print("📋 Clientes con compras above del promedio:")
print(df_subconsulta)

In [17]:
# Crear vista de resumen de ventas por cliente
query = '''
CREATE VIEW IF NOT EXISTS vw_resumen_ventas AS
SELECT
    c.nombre,
    c.ciudad,
    COUNT(p.id_pedido) as num_pedidos,
    ROUND(SUM(p.total), 2) as total_comprado,
    ROUND(AVG(p.total), 2) as ticket_promedio
FROM clientes c
LEFT JOIN pedidos p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente;
'''
conn.execute(query)
conn.commit()
print("✅ Vista vw_resumen_ventas creada exitosamente")

✅ Vista vw_resumen_ventas creada exitosamente


In [18]:
# Consultar la vista
df_vista = pd.read_sql_query('SELECT * FROM vw_resumen_ventas ORDER BY total_comprado DESC;', conn)
print("📋 VISTA: Resumen de ventas por cliente:")
print(df_vista)

📋 VISTA: Resumen de ventas por cliente:
             nombre    ciudad  num_pedidos  total_comprado  ticket_promedio
0      María García      Lima            4          4280.0           1070.0
1         Ana López     Cusco            2          3450.0           1725.0
2       Carmen Ruiz      Lima            1          2700.0           2700.0
3        Juan Pérez  Arequipa            2          1150.0            575.0
4    Laura Martínez  Trujillo            1          1050.0           1050.0
5       Rosa Torres  Arequipa            1          1050.0           1050.0
6  Carlos Rodríguez      Lima            2           580.0            290.0
7     Pedro Sánchez      Lima            1           450.0            450.0
8     Miguel Flores     Cusco            0             NaN              NaN
9         José Díaz  Trujillo            0             NaN              NaN


In [ ]:
# Crear vista de resumen de ventas por cliente
setup_database()  # garantiza que la BD esté lista
with sqlite3.connect(DB_FILE) as conn:
    conn.execute("""
        CREATE VIEW IF NOT EXISTS vw_resumen_ventas AS
        SELECT
            c.nombre,
            c.ciudad,
            COUNT(p.id_pedido)     as num_pedidos,
            ROUND(SUM(p.total), 2) as total_comprado,
            ROUND(AVG(p.total), 2) as ticket_promedio
        FROM clientes c
        LEFT JOIN pedidos p ON c.id_cliente = p.id_cliente
        GROUP BY c.id_cliente
    """)
print("✅ Vista vw_resumen_ventas creada exitosamente")

# Consultar la vista
df_vista = sql('SELECT * FROM vw_resumen_ventas ORDER BY total_comprado DESC;')
print("📋 VISTA: Resumen de ventas por cliente:")
print(df_vista)

# Filtrar usando la vista
df_clientes_top = sql('''
SELECT * FROM vw_resumen_ventas
WHERE total_comprado > 1000
ORDER BY total_comprado DESC;
''')
print("📋 Clientes TOP (más de S/1000 en compras):")
print(df_clientes_top)

**Contexto:** El gerente comercial necesita un informe con los 5 clientes más valiosos para ofrecerles beneficios exclusivos.

In [20]:
# Consulta para el TOP 5 de clientes
query = '''
SELECT
    c.nombre,
    c.ciudad,
    COUNT(p.id_pedido) as num_pedidos,
    ROUND(SUM(p.total), 2) as total_comprado,
    ROUND(AVG(p.total), 2) as ticket_promedio
FROM clientes c
INNER JOIN pedidos p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente
ORDER BY total_comprado DESC
LIMIT 5;
'''
df_top5 = pd.read_sql_query(query, conn)
print("🏆 TOP 5 CLIENTES:")
print(df_top5)

🏆 TOP 5 CLIENTES:
         nombre    ciudad  num_pedidos  total_comprado  ticket_promedio
0  María García      Lima            4          4280.0           1070.0
1     Ana López     Cusco            2          3450.0           1725.0
2   Carmen Ruiz      Lima            1          2700.0           2700.0
3    Juan Pérez  Arequipa            2          1150.0            575.0
4   Rosa Torres  Arequipa            1          1050.0           1050.0


**Pregunta A: Escribe una consulta SQL que identifique el TOP 5 de clientes por monto total de compras.**

**Respuesta:**

La consulta anterior muestra el TOP 5. Copia y pega la consulta aquí con tus comentarios.

# Consulta para el TOP 5 de clientes
query = '''
SELECT
    c.nombre,
    c.ciudad,
    COUNT(p.id_pedido)     as num_pedidos,
    ROUND(SUM(p.total), 2) as total_comprado,
    ROUND(AVG(p.total), 2) as ticket_promedio
FROM clientes c
INNER JOIN pedidos p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente
ORDER BY total_comprado DESC
LIMIT 5;
'''
df_top5 = sql(query)
print("🏆 TOP 5 CLIENTES:")
print(df_top5)

In [21]:
# Identificar clientes sin compras
query = '''
SELECT c.nombre, c.ciudad, c.email
FROM clientes c
LEFT JOIN pedidos p ON c.id_cliente = p.id_cliente
WHERE p.id_pedido IS NULL;
'''
df_sin_compras = pd.read_sql_query(query, conn)
print("📋 Clientes sin compras:")
print(df_sin_compras)

📋 Clientes sin compras:
          nombre    ciudad             email
0  Miguel Flores     Cusco  miguel@email.com
1      José Díaz  Trujillo    jose@email.com


**Pregunta C: Identifica qué clientes no han realizado ninguna compra. ¿Qué tipo de JOIN necesitas?**

**Respuesta:**

_Escribe tu respuesta aquí..._

In [ ]:
# Identificar clientes sin compras
query = '''
SELECT c.nombre, c.ciudad, c.email
FROM clientes c
LEFT JOIN pedidos p ON c.id_cliente = p.id_cliente
WHERE p.id_pedido IS NULL;
'''
df_sin_compras = sql(query)
print("📋 Clientes sin compras:")
print(df_sin_compras)

**Pregunta D: Si quisieras analizar las ventas por categoría de producto, ¿qué tablas necesitarías relacionar?**

**Respuesta:**

_Escribe tu respuesta aquí..._

In [ ]:
# Ventas por categoría de producto
query = '''
SELECT
    pr.categoria,
    COUNT(DISTINCT d.id_pedido)                   as num_pedidos,
    SUM(d.cantidad)                               as unidades_vendidas,
    ROUND(SUM(d.cantidad * d.precio_unitario), 2) as ingresos_totales
FROM productos pr
INNER JOIN detalles_pedido d ON pr.id_producto = d.id_producto
GROUP BY pr.categoria
ORDER BY ingresos_totales DESC;
'''
df_categoria = sql(query)
print("📋 VENTAS POR CATEGORÍA:")
print(df_categoria)

**Pregunta E: ¿Qué insights puedes obtener de la vista vw_ventas_por_ciudad?**

**Respuesta:**

_Escribe tu respuesta aquí..._

# Crear vista de ventas por ciudad
setup_database()
with sqlite3.connect(DB_FILE) as conn:
    conn.execute("""
        CREATE VIEW IF NOT EXISTS vw_ventas_por_ciudad AS
        SELECT
            c.ciudad,
            COUNT(DISTINCT c.id_cliente) as num_clientes,
            COUNT(p.id_pedido)           as num_pedidos,
            ROUND(SUM(p.total), 2)       as total_ventas
        FROM clientes c
        LEFT JOIN pedidos p ON c.id_cliente = p.id_cliente
        GROUP BY c.ciudad
    """)

# Consultar la vista
df_ciudad = sql('SELECT * FROM vw_ventas_por_ciudad ORDER BY total_ventas DESC;')
print("📋 VENTAS POR CIUDAD:")
print(df_ciudad)

Escribe tus conclusiones sobre lo aprendido en este laboratorio:

**1.** _Escribe tu primera conclusión aquí..._

**2.** _Escribe tu segunda conclusión aquí..._

**3.** _Escribe tu tercera conclusión aquí..._

In [24]:
# Cerrar conexión
conn.close()
print("✅ Conexión cerrada")

✅ Conexión cerrada


---
**© 2026 - Fundamentos de Gestión de Datos - TECSUP**  
**Docente:** Pilar Rocío Sayán Mejía